# TQP 본실험 v1.0: 예시 순서 섞기 (탐색적 민감도)

HW1 계획의 탐색적 민감도 분석입니다. Balanced와 Counterbalanced의 예시 순서를 seed로 고정한 무작위 순서 2가지로 바꿔 P와 B2를 다시 생성하고, 원래 순서와 비교합니다. 결과는 `main_v1/shuffle/`에 저장됩니다.

In [1]:
# 1-1. 설정
import os, sys, json, time, random, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/drive/MyDrive/NLP/tqp_repo')            # 코드 (Drive)
OUT = Path('/content/drive/MyDrive/NLP/AV2_TQP_TEG_Results/main_v1') # 결과 (Drive)
CACHE_ROOT = Path('/content/drive/MyDrive/NLP/AV2_TEG_Pilot_Data')  # AV2 로그 캐시 (파일럿과 공유)
FREEFORM_CSV = REPO / 'data' / 'freeform.csv'                  # 본인 검토를 마친 자유 문장
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
BATCH = 16
SEED = 2026
N_TRAIN_LOGS = 150
OUT.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels', 'boto3', 'openpyxl'], check=True)
sys.path.insert(0, str(REPO))
import importlib, numpy as np, pandas as pd
import tqp_teg as T, tqp_nl as N, tqp_eval as E, tqp_run as R, tqp_stats as S
for m in (T, N, E, R, S): importlib.reload(m)
print('engine', T.VERSION, '| detector', type(T.DEFAULT_DET).__name__, '| out', OUT)

Mounted at /content/drive
engine 0.4.0 | detector DetectorConfigV04 | out /content/drive/MyDrive/NLP/AV2_TQP_TEG_Results/main_v1


In [2]:
# 1-2. 평가 데이터 고정: 개발 세트(288), 평가 템플릿(864), 자유 문장 + 누출 검사 + 해시
dev = E.build_dev_set()
tpl = E.build_eval_templates()
free = E.load_freeform(FREEFORM_CSV) if FREEFORM_CSV.exists() else None
for df, p in ((dev, 'd'), (tpl, 't')) + (((free, 'f'),) if free is not None else ()):
    df['sid'] = [f'{p}{i:04d}' for i in range(len(df))]
evalset = pd.concat([tpl] + ([free] if free is not None else []), ignore_index=True)
leak = E.leakage_report(evalset)
assert leak.empty, leak
assert not set(evalset.sentence) & set(dev.sentence)
manifest = {'engine': T.VERSION, 'model': MODEL_ID, 'seed': SEED,
            'dev_sha256': E.frame_hash(dev), 'eval_tpl_sha256': E.frame_hash(tpl),
            'eval_free_sha256': E.frame_hash(free) if free is not None else None,
            'prompts_sha256': E.prompts_hash(),
            'n': {'dev': len(dev), 'eval_tpl': len(tpl), 'eval_free': 0 if free is None else len(free)},
            'clusters': {'dev': dev.cluster.nunique(), 'eval_tpl': tpl.cluster.nunique(),
                         'eval_free': 0 if free is None else free.cluster.nunique()}}
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=1))
for name, df in (('dev', dev), ('eval_tpl', tpl)) + ((('eval_free', free),) if free is not None else ()):
    df.to_csv(OUT / f'data_{name}.csv', index=False)
print(json.dumps(manifest, indent=1))
print(evalset.groupby(['split', 'gold_relation', 'order_type']).size())

{
 "engine": "0.4.0",
 "model": "Qwen/Qwen2.5-1.5B-Instruct",
 "seed": 2026,
 "dev_sha256": "7102eabaa6c3090e4b7dfba01f87bdf6ff6b0de4d192a1e7a32ccbdc51579740",
 "eval_tpl_sha256": "1e4209b602e31a58cf0024ace9ac91969785706733715182ac427ec22c7ea275",
 "eval_free_sha256": "64ff02138a8ec7c322156300a7c7c99c464152449f23ee735e3c4b5f65d6dacc",
 "prompts_sha256": "987707e71dd130314bcfbd8fb1795f1795d86b89f0eae84fbfec0d208c6963c7",
 "n": {
  "dev": 288,
  "eval_tpl": 864,
  "eval_free": 50
 },
 "clusters": {
  "dev": 72,
  "eval_tpl": 96,
  "eval_free": 11
 }
}
split      gold_relation  order_type  
eval_free  THEN           congruent        19
                          incongruent      16
           WHILE          simultaneous     15
eval_tpl   THEN           congruent       432
                          incongruent     324
           WHILE          simultaneous    108
dtype: int64


In [3]:
# 2. 생성: 조건 2 × 방법 2 × 순서 2 (평가 세트 914문장, T4)
import torch
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 바꾸십시오.'
model, tok, device = N.load_model(MODEL_ID)
SH = OUT / 'shuffle'
ORDERS = {}
for cond in ('balanced', 'counterbalanced'):
    n = len(N.VARIANTS[cond]['examples'])
    base = list(range(n))
    used = [base]
    rng = random.Random(SEED * 10 + n)
    for k in (1, 2):
        perm = base[:]
        while perm in used:
            rng.shuffle(perm)
        used.append(perm[:])
        ORDERS[(cond, k)] = perm
print(ORDERS)
splits = [('eval_tpl', tpl)] + ([('eval_free', free)] if free is not None else [])
raws = {}
t0 = time.time()
for (cond, k), perm in ORDERS.items():
    for method in N.METHODS:
        for split, data in splits:
            raws[(cond, k, method, split)] = R.generate_cell(model, tok, device, SH, split, method, cond, data,
                                                             batch_size=BATCH, example_order=perm, tag=f'_order{k}')
print(f'done in {(time.time() - t0) / 60:.1f} min')

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{('balanced', 1): [2, 0, 3, 1], ('balanced', 2): [0, 1, 3, 2], ('counterbalanced', 1): [5, 3, 1, 4, 2, 0], ('counterbalanced', 2): [2, 0, 5, 3, 1, 4]}
  generated eval_tpl/P/balanced_order1: 864 in 179s -> raw_eval_tpl_P_balanced_order1_e3ee83b20edfb8dd.csv
  generated eval_free/P/balanced_order1: 50 in 12s -> raw_eval_free_P_balanced_order1_f63f63d253ad1833.csv
  generated eval_tpl/B2/balanced_order1: 864 in 544s -> raw_eval_tpl_B2_balanced_order1_ac77cd3915fda156.csv
  generated eval_free/B2/balanced_order1: 50 in 35s -> raw_eval_free_B2_balanced_order1_4549cb8e59d7b63f.csv
  generated eval_tpl/P/balanced_order2: 864 in 179s -> raw_eval_tpl_P_balanced_order2_78a8064f682cefcb.csv
  generated eval_free/P/balanced_order2: 50 in 12s -> raw_eval_free_P_balanced_order2_7bcc8502606d9b6a.csv
  generated eval_tpl/B2/balanced_order2: 864 in 545s -> raw_eval_tpl_B2_balanced_order2_a8d1aa4e0b28794b.csv
  generated eval_free/B2/balanced_order2: 50 in 38s -> raw_eval_free_B2_balanced_order2_7a9c1c

In [4]:
# 3. 채점과 비교: 원래 순서(order0) vs 섞은 순서(order1, order2)
base_scored = pd.read_csv(OUT / 'scored_all.csv', low_memory=False)
base_ev = base_scored[base_scored.split.isin(['eval_tpl', 'eval_free']) & base_scored.condition.isin(['balanced', 'counterbalanced'])].copy()
base_ev['order'] = 0
rows = [base_ev]
for (cond, k, method, split), raw in raws.items():
    data = dict(splits)[split]
    sc = R.score_cell(data, raw, method, cond)
    sc['order'] = k
    rows.append(sc)
allsc = pd.concat(rows, ignore_index=True)
allsc.to_csv(SH / 'scored_shuffle.csv', index=False)
res = []
for cond in ('balanced', 'counterbalanced'):
    for k in (0, 1, 2):
        sub = allsc[(allsc.condition == cond) & (allsc.order == k)]
        d = S.paired_effect(sub, sub.method == 'P', sub.method == 'B2')
        res.append({'condition': cond, 'order': k, 'perm': str(ORDERS.get((cond, k), 'original')),
                    'Acc_bal_P': d['acc_a'], 'Acc_bal_B2': d['acc_b'], 'D': d['D'], 'CI_low': d['ci_low'],
                    'CI_high': d['ci_high'], 'p_perm': d['p_perm'],
                    'P_THEN': sub[(sub.method == 'P') & (sub.gold_relation == 'THEN')].interp_correct.mean(),
                    'P_WHILE': sub[(sub.method == 'P') & (sub.gold_relation == 'WHILE')].interp_correct.mean(),
                    'B2_THEN': sub[(sub.method == 'B2') & (sub.gold_relation == 'THEN')].interp_correct.mean(),
                    'B2_WHILE': sub[(sub.method == 'B2') & (sub.gold_relation == 'WHILE')].interp_correct.mean(),
                    'P_valid': sub[sub.method == 'P'].valid.mean()})
res = pd.DataFrame(res)
res.to_csv(SH / 'shuffle_summary.csv', index=False)
(SH / 'shuffle_summary.json').write_text(res.to_json(orient='records', indent=1))
print(res.round(3).to_string(index=False))

      condition  order               perm  Acc_bal_P  Acc_bal_B2      D  CI_low  CI_high  p_perm  P_THEN  P_WHILE  B2_THEN  B2_WHILE  P_valid
       balanced      0           original      0.599       0.456  0.144   0.049    0.238   0.026   0.540    0.659    0.578     0.333    0.834
       balanced      1       [2, 0, 3, 1]      0.464       0.455  0.009  -0.087    0.102   0.904   0.269    0.659    0.633     0.276    0.869
       balanced      2       [0, 1, 3, 2]      0.623       0.450  0.173   0.087    0.259   0.004   0.588    0.659    0.575     0.325    0.847
counterbalanced      0           original      0.431       0.333  0.099   0.043    0.150   0.198   0.326    0.537    0.641     0.024    0.833
counterbalanced      1 [5, 3, 1, 4, 2, 0]      0.456       0.465 -0.010  -0.075    0.060   0.819   0.407    0.504    0.565     0.366    0.844
counterbalanced      2 [2, 0, 5, 3, 1, 4]      0.597       0.387  0.210   0.157    0.255   0.011   0.544    0.650    0.718     0.057    0.846
